# 08 · PRDetect V4 Abla2 trên HC3

Bật GPU và Internet. Notebook lấy mã từ nhánh `version4` trên GitHub. Nhánh V3 train với `struct=0`, `aug=swap` và EMA. Nhánh A1 giữ công thức train của notebook 07. Điểm cuối là tích hai xác suất human với ngưỡng 0,5.

Để so sánh theo từng seed với V4, đặt `A1_CHECKPOINT` thành đường dẫn tới checkpoint A1 cùng seed của notebook 07 hoặc ZIP checkpoint do notebook 07 xuất ra. Khi đó chỉ train nhánh V3. Nếu để `None`, notebook train cả hai nhánh.

Ô cuối test ba tập như notebook 07 gồm HC3 sạch, GPT-3.5 Mixed sạch và HC3 Replace 25%.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
from zipfile import ZipFile
from IPython.display import FileLink, display

WORK = Path('/kaggle/working/prdetect_v4_abla2')
REPO = WORK / 'PRDetect-Research'
REPO_URL = 'https://github.com/tphan1420/PRDetect-Research.git'
BRANCH = 'version4'
WORK.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)],
                   check=True)
else:
    subprocess.run(['git', 'fetch', '--depth', '1', REPO_URL, BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO, check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print(f'PRDetect {BRANCH}: {commit}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-v3-eval.txt'),
                'torch-geometric'], check=True)
import spacy
if not spacy.util.is_package('en_core_web_sm'):
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
import torch
assert torch.cuda.is_available(), 'Bật GPU trong Kaggle Settings'
print(torch.cuda.get_device_name(0))

In [ ]:
SEED = 2026
V3_EPOCHS = 10
A1_EPOCHS = 10
BATCH_SIZE = 1
MICRO_BATCH_SIZE = 1
VAL_BATCH_SIZE = 64
A1_BATCH = 4
A1_LR = 0.0001
# Optional: path to a1_seed2026.pt or the v4_hc3_seed2026.zip from notebook 07.
A1_CHECKPOINT = None
CHECKPOINT_DIR = WORK / 'checkpoints'
FEATURE_CACHE = WORK / 'feature_cache'

In [ ]:
cmd = [sys.executable, 'train_v4.py', '--seed', str(SEED),
       '--output-dir', str(CHECKPOINT_DIR), '--feature-cache', str(FEATURE_CACHE),
       '--v3-epochs', str(V3_EPOCHS), '--v3-struct', '0', '--a1-epochs', str(A1_EPOCHS),
       '--batch-size', str(BATCH_SIZE), '--micro-batch-size', str(MICRO_BATCH_SIZE),
       '--val-batch-size', str(VAL_BATCH_SIZE), '--a1-batch', str(A1_BATCH), '--a1-lr', str(A1_LR)]
if A1_CHECKPOINT is not None:
    a1_path = Path(A1_CHECKPOINT)
    if a1_path.suffix.lower() == '.zip':
        name = f'a1_seed{SEED}.pt'
        with ZipFile(a1_path) as reference:
            if name not in reference.namelist():
                raise ValueError(f'{a1_path} không chứa {name}')
            a1_path = WORK / name
            a1_path.write_bytes(reference.read(name))
    cmd.extend(['--a1-checkpoint', str(a1_path)])

subprocess.run(cmd, cwd=REPO, check=True)
archive = CHECKPOINT_DIR / f'seed{SEED}' / f'v4_abla2_hc3_seed{SEED}.zip'
print(archive)
display(FileLink(str(archive)))

In [ ]:
OUT = WORK / f'abla2_three_tests_seed{SEED}'
subprocess.run([sys.executable, 'v4.py', 'evaluate', '--suite', 'three',
                '--checkpoint-dir', str(CHECKPOINT_DIR), '--seeds', str(SEED), '--batch-size', '1',
                '--device', 'cuda', '--output-dir', str(OUT)], cwd=REPO, check=True)
import pandas as pd
results = pd.read_csv(OUT / 'results_by_seed.csv')
expected = {('table2', 'Original'), ('table3', 'Original'), ('table4', 'Replace 25%')}
if len(results) != 3 or set(zip(results['table'], results['condition'])) != expected:
    raise ValueError('Kết quả không khớp ba tập test của notebook 07.')
display(results)
archive = shutil.make_archive(str(OUT), 'zip', OUT.parent, OUT.name)
display(FileLink(archive))